# Feature Engineering

Companion notebook for the [Feature Engineering lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/01-feature-engineering).

**The idea in one sentence.** The model is only as good as its features: **scaling** must
match the data (robust scalers for outliers), **categorical encodings** trade expressiveness
against leakage, and the single deadliest mistake is **fitting any transform on data that
includes the test set** — which silently leaks the future and inflates offline metrics.

What we build and verify:

- **Scaling:** Standard/MinMax explode under outliers; **RobustScaler** (median + IQR) does
  not.
- **Encoding:** one-hot, frequency, target — and why naive **target encoding leaks the
  label**.
- **The leakage trap:** fit the scaler inside a `Pipeline` on the *training fold only*.

We run all three, **validate the robust scaler and the leakage guard**, then cover the
gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Feature scaling: Standard vs MinMax vs Robust

In [ ]:
# Generate feature with outliers (income in thousands)
normal_incomes = np.random.normal(50, 15, 200)
outlier_incomes = np.array([200, 250, 300, 400])  # very high earners
X = np.concatenate([normal_incomes, outlier_incomes])

scalers = {
    'Original': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, scaler) in zip(axes, scalers.items()):
    if scaler is None:
        data = X
    else:
        data = scaler.fit_transform(X.reshape(-1, 1)).ravel()
    ax.hist(data, bins=30, color='#6366f1', alpha=0.8, edgecolor='#4f46e5')
    # Mark outliers
    if scaler is None:
        out_vals = outlier_incomes
    else:
        out_vals = scaler.transform(outlier_incomes.reshape(-1, 1)).ravel()
    ax.axvline(out_vals.mean(), color='#ef4444', linestyle='--', alpha=0.8, label=f'Outlier mean')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Value')
    if name != 'Original':
        ax.legend(fontsize=8, facecolor='#1a1d27', edgecolor='#2a2d3a')

plt.suptitle('Effect of scalers on income distribution with outliers', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\nRobustScaler is resistant to outliers:")
rs = RobustScaler().fit(X.reshape(-1, 1))
print(f"  Median: {rs.center_[0]:.1f}, IQR: {rs.scale_[0]:.1f}")

### Validate: RobustScaler resists outliers; the mean does not

Standard/MinMax scaling center on the **mean**, which a few extreme earners drag upward.
RobustScaler centers on the **median** and scales by the IQR, so its center stays near the
bulk of the data. We confirm the robust center is closer to the clean (outlier-free) median
than the mean is.

In [ ]:
ss = StandardScaler().fit(X.reshape(-1, 1))
rs = RobustScaler().fit(X.reshape(-1, 1))
clean_median = np.median(normal_incomes)
print(f'clean (no-outlier) median : {clean_median:.1f}')
print(f'StandardScaler center (mean)   : {ss.mean_[0]:.1f}  (pulled up by outliers)')
print(f'RobustScaler center (median)   : {rs.center_[0]:.1f}  (stays with the bulk)')
assert abs(rs.center_[0] - clean_median) < abs(ss.mean_[0] - clean_median), 'robust center resists outliers'
print('\n✅ RobustScaler (median/IQR) is the right choice when the feature has heavy outliers')

## Categorical encoding: comparing strategies

In [ ]:
import pandas as pd

# Simulate a dataset with a categorical feature and binary target
np.random.seed(0)
n = 1000
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']
city_rates = {'New York': 0.7, 'Los Angeles': 0.5, 'Chicago': 0.4, 'Houston': 0.3, 'Phoenix': 0.2}

city_col = np.random.choice(cities, n)
target = np.array([np.random.binomial(1, city_rates[c]) for c in city_col])

df = pd.DataFrame({'city': city_col, 'target': target})

# One-hot encoding
ohe = pd.get_dummies(df['city'], prefix='city')

# Frequency encoding
freq_enc = df['city'].map(df['city'].value_counts(normalize=True))

# Target encoding (DANGEROUS without cross-fold — showing the issue)
global_mean = df['target'].mean()
target_enc_naive = df.groupby('city')['target'].mean()
df['target_enc'] = df['city'].map(target_enc_naive)

print("Encoding comparison for 'city' feature:")
print("\nOne-hot (first 3 rows):")
print(pd.concat([df[['city', 'target']], ohe], axis=1).head(3).to_string())
print("\nTarget encoding (city → mean target):")
print(target_enc_naive.sort_values(ascending=False))

## Data leakage: the scaler trap

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_data, y_data = make_classification(n_samples=1000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# ❌ WRONG: fit scaler on full dataset before split
scaler_leaky = StandardScaler()
X_all_scaled = scaler_leaky.fit_transform(X_data)  # uses test set stats!
X_tr_leaky = X_all_scaled[:800]
X_te_leaky = X_all_scaled[800:]
clf_leaky = LogisticRegression(max_iter=500)
clf_leaky.fit(X_tr_leaky, y_tr)
acc_leaky = clf_leaky.score(X_te_leaky, y_te)

# ✅ CORRECT: fit scaler on training set only
pipe_correct = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=500))
])
pipe_correct.fit(X_tr, y_tr)  # scaler only sees X_tr
acc_correct = pipe_correct.score(X_te, y_te)

print(f"Leaky scaler (fit on full data):   {acc_leaky:.4f}")
print(f"Correct Pipeline (fit on train):   {acc_correct:.4f}")
print(f"Difference: {abs(acc_leaky - acc_correct):.4f} ({'leaky higher' if acc_leaky > acc_correct else 'correct higher'})")

### Validate: the leak is fitting the scaler on data that includes the test set

The correct pipeline fits its scaler on the **training fold only**; the leaky version fit on
the full dataset, so its statistics were contaminated by the test rows. We confirm the
pipeline's scaler used only training statistics, while the leaky scaler did not.

In [ ]:
fitted = pipe_correct.named_steps['scaler']
print(f'pipeline scaler mean == X_tr mean ? {np.allclose(fitted.mean_, X_tr.mean(axis=0))}')
print(f'leaky   scaler mean == X_tr mean ? {np.allclose(scaler_leaky.mean_, X_tr.mean(axis=0))}')
assert np.allclose(fitted.mean_, X_tr.mean(axis=0)), 'the correct pipeline scales using TRAIN stats only'
assert not np.allclose(scaler_leaky.mean_, X_tr.mean(axis=0)), 'the leaky scaler saw the full data (incl. test)'
print('\n✅ any transform (scaler, encoder, imputer) must be fit inside the CV fold — never on all data')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **wrong scaler** | Standard/MinMax distort under outliers; use RobustScaler (verified) |
| **fit on full data** | any transform fit before the split leaks the test set (verified) |
| **naive target encoding** | leaks the label; a random feature looks predictive (demo) |
| **one-hot high cardinality** | thousands of sparse columns; prefer frequency/target (out-of-fold) |
| **collinear features** | redundant features inflate variance; drop or regularize |

Demo: naive target encoding turns a random ID into a fake predictor.

In [ ]:
# The subtlest encoding gotcha: NAIVE target encoding leaks the label. If you replace each
# category with the mean target of its own rows (computed on the same data), a completely
# random, high-cardinality ID column becomes 'predictive' — pure leakage, no real signal.
import pandas as pd
rng = np.random.default_rng(0)
n = 600
yy = rng.integers(0, 2, n)
ids = rng.integers(0, n // 3, n)          # a random ID with NO relationship to y (~3 rows each)
d = pd.DataFrame({'id': ids, 'y': yy})
enc = d['id'].map(d.groupby('id')['y'].mean())   # naive target encoding on the SAME rows
corr = np.corrcoef(enc, yy)[0, 1]
print(f'correlation of a RANDOM id, naively target-encoded, with the target: {corr:.2f}')
assert corr > 0.4, 'naive target encoding makes a useless feature look predictive -> leakage'
print('A random column now "predicts" the label. Real target encoding needs out-of-fold means')
print('(K-fold / leave-one-out) so a row never sees its own label in its encoding.')

## ✏️ Your turn

### Exercise 1: Detect target leakage

Given a feature matrix and target, identify which features are likely target leakage by checking temporal constraints.

In [ ]:
def detect_high_correlation_features(X, y, threshold=0.9):
    """
    Find features with suspiciously high correlation with the target.
    High correlation with target may indicate target leakage.
    
    Args:
        X: np.ndarray (n_samples, n_features)
        y: np.ndarray (n_samples,) binary target
        threshold: float, correlation threshold to flag as suspicious
    Returns:
        list of int: feature indices with |correlation| > threshold
    """
    # TODO(you): compute Pearson correlation between each feature and y
    # Return indices where |correlation| > threshold
    pass


# Create test data: 2 normal features + 1 leaky (highly correlated with target)
np.random.seed(5)
n = 200
y_test = np.random.binomial(1, 0.5, n)
X_test = np.column_stack([
    np.random.randn(n),          # feature 0: unrelated
    np.random.randn(n),          # feature 1: unrelated  
    y_test + np.random.randn(n) * 0.05,  # feature 2: leaky (almost = target)
])

leaky = detect_high_correlation_features(X_test, y_test, threshold=0.9)
print(f"Suspicious features (|corr| > 0.9): {leaky}")

In [ ]:
leaky = detect_high_correlation_features(X_test, y_test, threshold=0.9)
assert leaky is not None, "Should return a list"
assert 2 in leaky, "Feature index 2 (the leaky one) should be flagged"
assert 0 not in leaky, "Feature 0 (unrelated) should not be flagged"

# Edge case: a zero-variance (constant) feature should never be flagged —
# its correlation is 0/epsilon ≈ 0, not NaN, thanks to the +1e-8 denominator guard.
X_const = np.column_stack([X_test, np.ones(n) * 7.0])
res_const = detect_high_correlation_features(X_const, y_test, threshold=0.9)
assert 3 not in res_const, "A constant feature has undefined correlation and must not be flagged"
assert 2 in res_const, "The leaky feature should still be flagged alongside the constant one"

# Edge case: a NEGATIVELY correlated leaky feature should also be flagged —
# leakage doesn't care about sign, we threshold on |correlation|.
X_neg = np.column_stack([X_test, -y_test.astype(float) + np.random.randn(n) * 0.05])
res_neg = detect_high_correlation_features(X_neg, y_test, threshold=0.9)
assert 3 in res_neg, "A strongly NEGATIVELY correlated feature should also be flagged"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def detect_high_correlation_features(X, y, threshold=0.9):
    suspicious = []
    y_centered = y - y.mean()
    for i in range(X.shape[1]):
        x_centered = X[:, i] - X[:, i].mean()
        corr = np.dot(x_centered, y_centered) / (
            np.linalg.norm(x_centered) * np.linalg.norm(y_centered) + 1e-8
        )
        if abs(corr) > threshold:
            suspicious.append(i)
    return suspicious
```
</details>

### Exercise 2: Implement frequency encoding

Replace each category with its frequency (proportion) in the training set.

In [ ]:
def frequency_encode(train_categories, test_categories):
    """
    Encode categories by their frequency in the training set.
    
    Args:
        train_categories: list/array of category labels (training set)
        test_categories: list/array of category labels (test set)
    Returns:
        tuple: (train_encoded, test_encoded) — arrays of float frequencies
               Unknown categories in test get frequency 0.0
    """
    # TODO(you): compute category frequencies from train_categories
    # Map train and test to those frequencies (unknown test categories → 0.0)
    pass


train_cats = ['A', 'B', 'A', 'C', 'A', 'B', 'C', 'A']  # A:4, B:2, C:2
test_cats = ['A', 'B', 'D']  # D is unseen

tr_enc, te_enc = frequency_encode(train_cats, test_cats)
print(f"Train encoded: {tr_enc}")
print(f"Test encoded:  {te_enc}")

In [ ]:
tr_enc, te_enc = frequency_encode(train_cats, test_cats)
assert tr_enc is not None, "Should return train encodings"
assert len(tr_enc) == len(train_cats), "Train encoding length mismatch"
assert abs(tr_enc[0] - 4/8) < 1e-6, "'A' frequency should be 4/8 = 0.5"
assert te_enc[-1] == 0.0, "Unseen category 'D' should get frequency 0.0"

# Edge case: a single-category training set — every train row gets frequency 1.0.
tr_single, te_single = frequency_encode(['A', 'A', 'A'], ['A', 'B'])
assert np.allclose(tr_single, 1.0), "A training set with one unique category should encode everything as 1.0"
assert te_single[0] == 1.0 and te_single[1] == 0.0, "Seen category -> 1.0, unseen category -> 0.0"

# Edge case: empty test set — should return an empty (not crashing) array.
tr_empty_test, te_empty_test = frequency_encode(['A', 'B'], [])
assert len(te_empty_test) == 0, "Empty test_categories should produce an empty encoding array"
assert len(tr_empty_test) == 2, "Train encoding should be unaffected by an empty test set"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def frequency_encode(train_categories, test_categories):
    n = len(train_categories)
    from collections import Counter
    counts = Counter(train_categories)
    freq_map = {cat: count / n for cat, count in counts.items()}
    train_enc = np.array([freq_map[c] for c in train_categories])
    test_enc = np.array([freq_map.get(c, 0.0) for c in test_categories])
    return train_enc, test_enc
```
</details>

---
### Exercise 3 — DML feature-transform bank (32, 84, 34, 112, 141)

Five small [Open-Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) problems
that all belong to the "prepare raw features for a model" family. Implement
each with plain NumPy/Python — no `sklearn`:

- DML `32` `generate-sorted-polynomial-features` → `polynomial_features_sorted(X, degree)`
  — every polynomial combination of the columns of a 2-D array up to `degree`,
  with each **row** sorted ascending.
- DML `84` `phi-transformation-for-polynomial-features` → `phi_transform(data, degree)`
  — a distinct, simpler 1-D variant: for a flat list of numbers, return
  `[x**0, x**1, ..., x**degree]` per element (unsorted). Returns `[]` if
  `degree < 0` or `data` is empty.
- DML `34` `one-hot-encoding-of-nominal-values` → `one_hot_encode(x, n_col=None)`
  — infer the column count from `max(x) + 1` when `n_col` isn't given.
- DML `112` `min-max-normalization-of-feature-values` → `min_max_normalize(x)`
  — scale a list to `[0, 1]`; a zero-variance list (`max == min`) maps to all
  zeros rather than dividing by zero.
- DML `141` `shift-and-scale-array-to-target-range` → `convert_range(values, c, d)`
  — remap a NumPy array (1-D or 2-D) from its own `[min, max]` to an arbitrary
  `[c, d]`.

In [ ]:
from itertools import combinations_with_replacement


def polynomial_features_sorted(X, degree):
    """DML 32. X: np.ndarray (n_samples, n_features). Returns ascending-sorted
    polynomial features (incl. the degree-0 "1" term) per row."""
    X = np.asarray(X, dtype=float)
    n_samples, n_features = X.shape
    # TODO(you): build every combination of column indices (with replacement)
    # for each degree 0..degree, take the product of those columns per row,
    # stack into a (n_samples, n_terms) array, then sort each row ascending.
    pass


def phi_transform(data, degree):
    """DML 84. data: flat list of floats. Returns [[x**0, ..., x**degree], ...]
    (unsorted, NOT the same transform as DML 32). Empty list if degree < 0."""
    # TODO(you): if degree < 0 return []; else return [x**p for p in 0..degree] per x
    pass


def one_hot_encode(x, n_col=None):
    """DML 34. x: 1-D int array. n_col inferred from max(x)+1 if not given."""
    x = np.asarray(x)
    # TODO(you): build an (len(x), n_col) zero matrix, set a 1 at (row, x[row])
    pass


def min_max_normalize(x):
    """DML 112. x: list of numbers -> list of floats in [0, 1].
    Zero-variance input (max == min) should return all zeros, not NaN."""
    # TODO(you): (v - min) / (max - min) per element, guarding max == min
    pass


def convert_range(values, c, d):
    """DML 141. Remap values (1-D or 2-D np.ndarray) from their own [min, max]
    to the target [c, d]. f(x) = c + (d - c) / (b - a) * (x - a)."""
    values = np.asarray(values, dtype=float)
    # TODO(you): compute a = values.min(), b = values.max(), apply the formula
    pass


# Smoke test against DML's own worked examples
print("polynomial_features_sorted:", polynomial_features_sorted(np.array([[2, 3], [3, 4], [5, 6]]), 2))
print("phi_transform:", phi_transform([1.0, 2.0], 2))
print("one_hot_encode:\n", one_hot_encode(np.array([0, 1, 2, 1, 0])))
print("min_max_normalize:", min_max_normalize([1, 2, 3, 4, 5]))
print("convert_range:", convert_range(np.array([0, 5, 10]), 2, 4))

In [ ]:
import math

def close(a, b, tol=1e-6):
    return abs(a - b) < tol

# --- DML 32: polynomial_features_sorted -------------------------------
out32 = polynomial_features_sorted(np.array([[2, 3], [3, 4], [5, 6]]), 2)
assert out32 is not None, "polynomial_features_sorted returned None"
expected32 = np.array([[1, 2, 3, 4, 6, 9], [1, 3, 4, 9, 12, 16], [1, 5, 6, 25, 30, 36]], dtype=float)
assert np.allclose(out32, expected32), f"DML 32 mismatch:\n{out32}"
# Edge case: single feature column, degree 1 -> just [1, x] sorted per row
single_feat = polynomial_features_sorted(np.array([[4.0], [-1.0]]), 1)
assert np.allclose(np.sort(single_feat, axis=1), single_feat), "Rows must already be ascending-sorted"
assert np.allclose(single_feat, np.array([[1.0, 4.0], [-1.0, 1.0]])), f"single-feature case: {single_feat}"

# --- DML 84: phi_transform ---------------------------------------------
assert phi_transform([], 2) == [], "Empty data should give an empty list"
assert phi_transform([1.0, 2.0], -1) == [], "Negative degree should give an empty list"
assert phi_transform([1.0, 2.0], 2) == [[1.0, 1.0, 1.0], [1.0, 2.0, 4.0]], f"DML 84 mismatch: {phi_transform([1.0, 2.0], 2)}"
assert phi_transform([2.0], 4) == [[1.0, 2.0, 4.0, 8.0, 16.0]], "Single-element data case failed"

# --- DML 34: one_hot_encode ---------------------------------------------
oh1 = one_hot_encode(np.array([0, 1, 2, 1, 0]))
assert np.allclose(oh1, np.array([[1,0,0],[0,1,0],[0,0,1],[0,1,0],[1,0,0]], dtype=float)), f"DML 34 mismatch: {oh1}"
oh2 = one_hot_encode(np.array([3, 1, 2, 1, 3]), 4)
assert np.allclose(oh2, np.array([[0,0,0,1],[0,1,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float)), f"DML 34 (explicit n_col) mismatch: {oh2}"
# Edge case: single-row input
oh_single = one_hot_encode(np.array([2]), 3)
assert np.allclose(oh_single, np.array([[0.0, 0.0, 1.0]])), "Single-row one-hot failed"

# --- DML 112: min_max_normalize ------------------------------------------
assert min_max_normalize([1, 2, 3, 4, 5]) == [0.0, 0.25, 0.5, 0.75, 1.0], "DML 112 basic case failed"
assert [round(v, 4) for v in min_max_normalize([30, 45, 56, 70, 88])] == [0.0, 0.2586, 0.4483, 0.6897, 1.0], \
    "DML 112 non-round-number case failed"
# Edge case: zero-variance list -> all zeros, not NaN/inf
zero_var = min_max_normalize([5, 5, 5, 5])
assert zero_var == [0.0, 0.0, 0.0, 0.0], f"Zero-variance input should map to all zeros, got {zero_var}"
# Edge case: single-element list -> also zero-variance by definition
assert min_max_normalize([7]) == [0.0], "Single-element input should map to [0.0]"

# --- DML 141: convert_range ----------------------------------------------
r1 = convert_range(np.array([0, 5, 10]), 2, 4)
assert np.allclose(r1, [2.0, 3.0, 4.0]), f"DML 141 basic case failed: {r1}"
seq = np.array([388, 242, 124, 384, 313, 277, 339, 302, 268, 392])
r2 = np.round(convert_range(seq, 0, 1), 6)
assert np.allclose(r2, [0.985075, 0.440299, 0.0, 0.970149, 0.705224, 0.570896, 0.802239, 0.664179, 0.537313, 1.0]), \
    f"DML 141 1-D case failed: {r2}"
seq2 = np.array([[2028, 4522], [1412, 2502], [3414, 3694], [1747, 1233], [1862, 4868]])
r3 = np.round(convert_range(seq2, 4, 8), 6)
expected3 = np.array([[4.874828, 7.619257], [4.196974, 5.396424], [6.4, 6.708116], [4.565612, 4.0], [4.69216, 8.0]])
assert np.allclose(r3, expected3), f"DML 141 2-D case failed: {r3}"
# Edge case: threshold at the exact min/max — endpoints must land exactly on c and d
assert close(convert_range(seq, 0, 1)[np.argmin(seq)], 0.0), "The minimum value must map exactly to c"
assert close(convert_range(seq, 0, 1)[np.argmax(seq)], 1.0), "The maximum value must map exactly to d"
# Edge case: single-value array (min == max) — our convention maps to the midpoint of [c, d]
single_val = convert_range(np.array([5.0]), 2.0, 4.0)
assert np.allclose(single_val, [3.0]), f"Single-value (zero-range) input should map to the midpoint, got {single_val}"

print("✅ Exercise 3 passed (DML 32, 84, 34, 112, 141)")

<details>
<summary>💡 Show solution</summary>

```python
from itertools import combinations_with_replacement


def polynomial_features_sorted(X, degree):
    X = np.asarray(X, dtype=float)
    n_samples, n_features = X.shape
    rows = []
    for i in range(n_samples):
        terms = []
        for d in range(degree + 1):
            for combo in combinations_with_replacement(range(n_features), d):
                val = 1.0
                for idx in combo:
                    val *= X[i, idx]
                terms.append(val)
        rows.append(sorted(terms))
    return np.array(rows)


def phi_transform(data, degree):
    if degree < 0 or len(data) == 0:
        return []
    return [[x**p for p in range(degree + 1)] for x in data]


def one_hot_encode(x, n_col=None):
    x = np.asarray(x)
    if n_col is None:
        n_col = int(x.max()) + 1
    out = np.zeros((len(x), n_col))
    for row, val in enumerate(x):
        out[row, val] = 1.0
    return out


def min_max_normalize(x):
    x = np.asarray(x, dtype=float)
    mn, mx = x.min(), x.max()
    if mx == mn:
        return [0.0] * len(x)
    return list((x - mn) / (mx - mn))


def convert_range(values, c, d):
    values = np.asarray(values, dtype=float)
    a, b = values.min(), values.max()
    if b == a:
        return np.full_like(values, (c + d) / 2)
    return c + (d - c) / (b - a) * (values - a)
```
</details>

## Key takeaways

- **Match the scaler to the data:** Standard/MinMax break under outliers; RobustScaler
  (median/IQR) does not (verified).
- **The leakage trap:** fit every transform inside the CV fold, on training rows only — a
  `Pipeline` enforces this (verified).
- **Encodings trade off:** one-hot is safe but wide; frequency is compact; **naive target
  encoding leaks the label** and must use out-of-fold means (demo).
- **Feature engineering is where most offline/online gaps are born** — audit every transform
  for leakage.